In [1]:
# IMPORTS
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

In [2]:
# CONTROL PANEL
seq_len = 12
R = 6371000  
epoch = 50
lr = 0.001
batch_size = 64

In [3]:
# FIX DATA
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")


cols_to_remove = [
    'cog_rad',
    'cog_sin',
    'cog_cos',
    'hdg_rad',
    'hdg_sin',
    'hdg_cos',
    'dlat',
    'dlon'
]

train_df = train_df.drop(columns=cols_to_remove)
test_df = test_df.drop(columns=cols_to_remove)


train_cog_rad = np.deg2rad(train_df["COG"])
train_heading_rad = np.deg2rad(train_df["HEADING"])

test_cog_rad = np.deg2rad(test_df["COG"])
test_heading_rad = np.deg2rad(test_df["HEADING"])


train_df["COG_cos"] = np.cos(train_cog_rad)
train_df["COG_sin"] = np.sin(train_cog_rad)

train_df["HEADING_cos"] = np.cos(train_heading_rad)
train_df["HEADING_sin"] = np.sin(train_heading_rad)


test_df["COG_cos"] = np.cos(test_cog_rad)
test_df["COG_sin"] = np.sin(test_cog_rad)

test_df["HEADING_cos"] = np.cos(test_heading_rad)
test_df["HEADING_sin"] = np.sin(test_heading_rad)

train_df["LAT_rad"] = np.deg2rad(train_df["LAT"])
train_df["LON_rad"] = np.deg2rad(train_df["LON"])

test_df["LAT_rad"] = np.deg2rad(test_df["LAT"])
test_df["LON_rad"] = np.deg2rad(test_df["LON"])

train_df["dy"] = (
    train_df.groupby("MMSI")["LAT_rad"].shift(-1) - train_df["LAT_rad"]
) * R

train_df["dx"] = (
    train_df.groupby("MMSI")["LON_rad"].shift(-1) - train_df["LON_rad"]
) * R * np.cos(train_df["LAT_rad"])


test_df["dy"] = (
    test_df.groupby("MMSI")["LAT_rad"].shift(-1) - test_df["LAT_rad"]
) * R

test_df["dx"] = (
    test_df.groupby("MMSI")["LON_rad"].shift(-1) - test_df["LON_rad"]
) * R * np.cos(test_df["LAT_rad"])

train_df = train_df.dropna(subset=["dx", "dy"])
test_df = test_df.dropna(subset=["dx", "dy"])

train_df = train_df.drop(columns=["LAT_rad","LON_rad"])
test_df = test_df.drop(columns=["LAT_rad","LON_rad"])

train_df["TIME"] = pd.to_datetime(train_df["TIME"])
test_df["TIME"] = pd.to_datetime(test_df["TIME"])

train_df = train_df.sort_values(["voyage_id", "TIME"]).reset_index(drop=True)
test_df = test_df.sort_values(["voyage_id", "TIME"]).reset_index(drop=True)

train_df["dlat"] = train_df.groupby("voyage_id")["LAT"].shift(-1) - train_df["LAT"]
train_df["dlon"] = train_df.groupby("voyage_id")["LON"].shift(-1) - train_df["LON"]

test_df["dlat"] = test_df.groupby("voyage_id")["LAT"].shift(-1) - test_df["LAT"]
test_df["dlon"] = test_df.groupby("voyage_id")["LON"].shift(-1) - test_df["LON"]

train_df = train_df.dropna(subset=["dlat", "dlon"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["dlat", "dlon"]).reset_index(drop=True)

train_df.loc[train_df.groupby("voyage_id").head(1).index, "dt"] = 0
test_df.loc[test_df.groupby("voyage_id").head(1).index, "dt"] = 0

train_df["dlat"] = train_df.groupby("voyage_id")["LAT"].diff()
train_df["dlon"] = train_df.groupby("voyage_id")["LON"].diff()

test_df["dlat"] = test_df.groupby("voyage_id")["LAT"].diff()
test_df["dlon"] = test_df.groupby("voyage_id")["LON"].diff()

train_df["dlat"] = train_df["dlat"].fillna(0)
train_df["dlon"] = train_df["dlon"].fillna(0)

test_df["dlat"] = test_df["dlat"].fillna(0)
test_df["dlon"] = test_df["dlon"].fillna(0)

In [5]:
#TCN SEQUENCE
def build_tcn_sequences(df, seq_len=seq_len):

    feature_cols = [
        "LAT",
        "LON",
        "SPEED",
        "dt",
        "COG_cos",
        "COG_sin",
        "HEADING_cos",
        "HEADING_sin"
    ]

    target_cols = ["dlat", "dlon"]

    X = []
    Y = []

    for voyage_id, group in df.groupby("voyage_id"):

        group = group.sort_values("TIME")

        features = group[feature_cols].values
        targets = group[target_cols].values

        for i in range(len(group) - seq_len):

            X.append(features[i:i + seq_len])
            Y.append(targets[i + seq_len])

    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float32)

    return X, Y

In [6]:
#LOAD TRAIN/TEST
X_train, Y_train = build_tcn_sequences(train_df, seq_len)
X_test, Y_test = build_tcn_sequences(test_df, seq_len)

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, X_train.shape[-1])
X_test_2d  = X_test.reshape(-1, X_test.shape[-1])

scaler.fit(X_train_2d)

X_train_scaled = scaler.transform(X_train_2d).reshape(X_train.shape)
X_test_scaled  = scaler.transform(X_test_2d).reshape(X_test.shape)

X_train = X_train_scaled.astype(np.float32)
X_test  = X_test_scaled.astype(np.float32)

X_train_t = torch.tensor(X_train)
Y_train_t = torch.tensor(Y_train)

X_test_t = torch.tensor(X_test)
Y_test_t = torch.tensor(Y_test)

train_dataset = TensorDataset(X_train_t, Y_train_t)
test_dataset = TensorDataset(X_test_t, Y_test_t)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [7]:
# TEMPORAL BLOCK
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.relu2 = nn.ReLU()

        self.downsample = nn.Conv1d(in_channels, out_channels, 1) \
            if in_channels != out_channels else None

    def forward(self, x):

        out = self.conv1(x)
        out = self.relu1(out)

        out = self.conv2(out)
        out = self.relu2(out)

        res = x if self.downsample is None else self.downsample(x)

        return out[:, :, :res.shape[2]] + res

In [8]:
# TCN BUILD
class TCN(nn.Module):
    def __init__(self, input_dim, output_dim):

        super().__init__()

        self.tcn = nn.Sequential(
            TemporalBlock(input_dim, 32, kernel_size=3, dilation=1),
            TemporalBlock(32, 64, kernel_size=3, dilation=2),
            TemporalBlock(64, 64, kernel_size=3, dilation=4)
        )

        self.fc = nn.Linear(64, output_dim)

    def forward(self, x):

        x = x.transpose(1, 2)

        y = self.tcn(x)

        y = y[:, :, -1]

        return self.fc(y)

In [9]:
#SETTING THE MODEL
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TCN(input_dim=8, output_dim=2).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

/home/jacobhardy/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [10]:
# TRAINING
for epoch in range(epoch):

    model.train()
    total_loss = 0

    for X_batch, Y_batch in train_loader:

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(pred, Y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch, "Loss:", total_loss / len(train_loader))

Epoch: 0 Loss: 0.0018939628780093411
Epoch: 1 Loss: 0.00013518846257407256
Epoch: 2 Loss: 0.00011789069133084198
Epoch: 3 Loss: 0.00011289928459559735
Epoch: 4 Loss: 0.00011133825774551842
Epoch: 5 Loss: 0.0001112828327007408
Epoch: 6 Loss: 0.00010328243180473302
Epoch: 7 Loss: 9.86466947686946e-05
Epoch: 8 Loss: 0.00010575952031179174
Epoch: 9 Loss: 0.00011861726364402689
Epoch: 10 Loss: 0.00010099439212463313
Epoch: 11 Loss: 9.231243410952568e-05
Epoch: 12 Loss: 9.612143900509332e-05
Epoch: 13 Loss: 9.403736225805943e-05
Epoch: 14 Loss: 9.292544824087859e-05
Epoch: 15 Loss: 9.66019810275748e-05
Epoch: 16 Loss: 0.00010299595173716824
Epoch: 17 Loss: 9.651287707259013e-05
Epoch: 18 Loss: 9.225083534632936e-05
Epoch: 19 Loss: 9.268079269586588e-05
Epoch: 20 Loss: 9.347827191029359e-05
Epoch: 21 Loss: 9.429580912087419e-05
Epoch: 22 Loss: 9.065201116153135e-05
Epoch: 23 Loss: 9.208819397840704e-05
Epoch: 24 Loss: 9.705884861576028e-05
Epoch: 25 Loss: 9.383346742084486e-05
Epoch: 26 Loss:

In [11]:
# EVALUATION
model.eval()

preds = []
targets = []
inputs = []

with torch.no_grad():

    for X_batch, Y_batch in test_loader:

        X_batch = X_batch.to(device)

        pred = model(X_batch).cpu()

        preds.append(pred)
        targets.append(Y_batch)
        inputs.append(X_batch.cpu())

preds = torch.cat(preds).numpy()
targets = torch.cat(targets).numpy()
inputs = torch.cat(inputs).numpy()

last_lat = inputs[:, -1, 0]
last_lon = inputs[:, -1, 1]

lat_pred = last_lat + preds[:,0]
lon_pred = last_lon + preds[:,1]

lat_true = last_lat + targets[:,0]
lon_true = last_lon + targets[:,1]

lat_error = lat_pred - lat_true
lon_error = lon_pred - lon_true

rmse = np.sqrt(np.mean(lat_error**2 + lon_error**2))
mae = np.mean(np.sqrt(lat_error**2 + lon_error**2))

lat_error_rad = np.radians(lat_error)
lon_error_rad = np.radians(lon_error)

lat_true_rad = np.radians(lat_true)

x = lon_error_rad * np.cos(lat_true_rad)
y = lat_error_rad

dist_m = R * np.sqrt(x**2 + y**2)

rmse_m = np.sqrt(np.mean(dist_m**2))
mae_m = np.mean(dist_m)

print("RMSE (meters):", rmse_m)
print("MAE (meters):", mae_m)

RMSE (meters): 837.518
MAE (meters): 452.19586
